# Nemotron Blackwell — Pipeline Smoke Test

**Before running:** set notebook settings and attach inputs.

| Setting | Value |
|---------|-------|
| Accelerator | **RTX PRO 6000 (Blackwell)** |
| Internet | OFF (forced on Blackwell) |

| Input | How to attach |
|-------|---------------|
| Competition data | Auto when created from competition page |
| Nemotron-30B | **Add Input → Models →** `metric/nemotron-3-nano-30b-a3b-bf16` |
| Training code | **Add Input → Datasets →** your `nemotron-kaggle-code` dataset |

The code dataset is tiny (~few MB). Publish once from a T4 notebook:
```python
!git clone https://github.com/gbose01/nemotron_kaggle.git /kaggle/working/nemotron_kaggle
%cd /kaggle/working/nemotron_kaggle
!python scripts/stage_code_only.py
# Upload /kaggle/temp/nemotron-kaggle-code as private dataset
```

This notebook runs **2 training steps** on inline examples to verify:
load model → train LoRA → pack `submission.zip`.

In [ ]:
import shutil
from pathlib import Path

print("Mounted inputs:")
for p in sorted(Path("/kaggle/input").iterdir()):
    print(f"  {p.name}")

code_src = next(Path("/kaggle/input").rglob("src/blackwell_env.py")).parent.parent
model_dir = next(
    cfg.parent
    for cfg in Path("/kaggle/input").rglob("config.json")
    if (cfg.parent / "model.safetensors.index.json").exists()
    or list(cfg.parent.glob("model-*.safetensors"))
)

shutil.copytree(code_src, "/kaggle/working/nemotron_kaggle", dirs_exist_ok=True)
print("\nModel:", model_dir)
print("Code: ", "/kaggle/working/nemotron_kaggle")

In [ ]:
!PYTHONPATH=src python3 /kaggle/working/nemotron_kaggle/scripts/setup_kaggle_inputs.py

## Verify GPU and packages

If `mamba_ssm` or `torch` is missing, stop here and use the bootstrap path in `kaggle_guide.md`.

In [ ]:
import torch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

missing = []
for pkg in ("transformers", "peft", "trl", "datasets", "mamba_ssm"):
    try:
        __import__(pkg)
        print(f"  {pkg}: OK")
    except ImportError:
        print(f"  {pkg}: MISSING")
        missing.append(pkg)

if missing:
    raise SystemExit(f"Missing packages: {missing}. See kaggle_guide.md bootstrap section.")

## Smoke test (~10 min)

2 steps, LoRA rank 8, builds `/kaggle/working/submission.zip`.

In [ ]:
%cd /kaggle/working/nemotron_kaggle
!PYTHONPATH=src python3 src/smoke_blackwell.py --max_steps 2 --lora_rank 8

In [ ]:
from IPython.display import FileLink

FileLink("/kaggle/working/submission.zip")

---

## Optional: submit smoke test

Use **Submit to Competition** with `submission.zip`. Score will be low — this only checks format.

---

## Full training (after smoke test passes)

Uncomment and run the cells below (~1 hour).

In [ ]:
# %cd /kaggle/working/nemotron_kaggle
# !PYTHONPATH=src python3 src/train_blackwell.py \
#   --config src/train_config.yaml \
#   --data data/sft_reasoning_dataset.jsonl \
#   --output_dir outputs/nemotron_lora_adapter

In [ ]:
# !python3 src/pack_submission.py \
#   --source outputs/nemotron_lora_adapter \
#   --output /kaggle/working/submission.zip
# !python3 src/verify_submission.py --submission /kaggle/working/submission.zip
# from IPython.display import FileLink
# FileLink("/kaggle/working/submission.zip")